# Credit Fraud Mentoring Walkthrough

Thin notebook: **only calls package functions**. Logic lives under `src/credit_fraud/`.

For the Optuna / Bayesian session, use [`02_optuna_bayesian.ipynb`](02_optuna_bayesian.ipynb) (step-by-step) and Streamlit → **Optuna lab**.

Prerequisite (repo root):

```bash
pip install -e ".[dev]"
```

In [1]:
from credit_fraud import config
from credit_fraud.data import feature_glossary, load_and_split
from credit_fraud.eda import run_eda
from credit_fraud.rank_plot import rank_and_plot
from credit_fraud.features import build_features
from credit_fraud.benchmark import run_benchmark
from credit_fraud.optimize import optimize_models
from credit_fraud.selection import select_best

print("TARGET_COL =", config.TARGET_COL)
print("MAX_ROWS   =", config.MAX_ROWS)
print("N_TRIALS   =", config.N_TRIALS)
print("DATA_PATH  =", config.DATA_PATH)
display(feature_glossary())  # interpretable column meanings

Matplotlib is building the font cache; this may take a moment.


TARGET_COL = default_next_month
MAX_ROWS   = None
N_TRIALS   = 15
DATA_PATH  = /Users/himanshujhamb/Desktop/Personal/Home_Assignment/Naresh/data/credit_default.csv


,feature,description
0,LIMIT_BAL,"Credit limit (NT dollars), individual + family..."
1,SEX,"Gender code: 1 = male, 2 = female."
2,EDUCATION,"1=graduate school, 2=university, 3=high school..."
3,MARRIAGE,"1=married, 2=single, 3=others."
4,AGE,Age in years.
5,PAY_0,"Repayment status in Sep 2005 (−1=pay duly, 1=1..."
6,PAY_2,Repayment status in Aug 2005 (same coding as P...
7,PAY_3,Repayment status in Jul 2005.
8,PAY_4,Repayment status in Jun 2005.
9,PAY_5,Repayment status in May 2005.


## 1) Load + stratified split

In [2]:
x_train, x_test, y_train, y_test, raw = load_and_split()
print("file:", config.DATA_PATH)
print("target:", y_train.name, "positive rate:", round(float(y_train.mean()), 4))
print("columns:", list(raw.columns))
x_train.shape, y_train.mean()

Loading local CSV: /Users/himanshujhamb/Desktop/Personal/Home_Assignment/Naresh/data/credit_default.csv
file: /Users/himanshujhamb/Desktop/Personal/Home_Assignment/Naresh/data/credit_default.csv
target: default_next_month positive rate: 0.2212
columns: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_next_month']


((24000, 23), np.float64(0.22120833333333334))

## 2) EDA (independent + dependent)

In [3]:
eda = run_eda(x_train, y_train, save=True)
display(eda.target_summary)
print("imbalance neg/pos =", eda.imbalance_ratio)
eda.figures["target_distribution"].show()
# Raw independent-variable distributions (dropdown)
eda.figures["raw_distributions"].show()

,default_next_month,count,rate
0,0,18691,0.78
1,1,5309,0.22


imbalance neg/pos = 3.52


## 3) Rank & plot

Use the **Feature** dropdown on the Plotly chart to switch independent variables.

**Dataset:** UCI Taiwan credit **default** (`LIMIT_BAL`, `AGE`, `PAY_*`, `BILL_AMT_*`, …) — see `docs/dataset_features.md` and `feature_glossary(...)`.

Also read before modelling:
- `docs/bayesian_optimization_optuna.md` (layman session script)
- `docs/xgboost_guide.md`
- `notebooks/02_optuna_bayesian.ipynb` (Optuna step by step)

In [4]:
ranks = rank_and_plot(x_train, y_train, save=True)
display(ranks.ranking.head(10))
print("Columns in X (from the dataset):", list(x_train.columns))

# Rank & plot only: dropdown over features — bars = n_obs, line = capture rate
ranks.figures["capture_dropdown"].show()

,feature,univariate_auc,mutual_info,rank_auc
0,PAY_0,0.69,0.07,1
1,PAY_2,0.64,0.05,2
2,PAY_3,0.63,0.03,3
3,LIMIT_BAL,0.62,0.02,4
4,PAY_4,0.61,0.03,5
5,PAY_AMT1,0.61,0.03,6
6,PAY_AMT2,0.60,0.02,7
7,PAY_5,0.60,0.03,8
8,PAY_AMT3,0.60,0.01,9
9,PAY_6,0.59,0.03,10


Columns in X (from the dataset): ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


## 4–5) Features + benchmark

In [5]:
x_train_fe, x_test_fe, builder = build_features(x_train, x_test, y_train)
bench = run_benchmark(x_train_fe, x_test_fe, y_train, y_test)
bench.metrics

{'majority': {'pr_auc': 0.22,
  'roc_auc': 0.5,
  'f1': 0.0,
  'precision': 0.0,
  'recall': 0.0},
 'logistic': {'pr_auc': 0.49,
  'roc_auc': 0.73,
  'f1': 0.5,
  'precision': 0.41,
  'recall': 0.64}}

## 6–7) Optuna (RF + XGBoost) → selection

Uses `config.N_TRIALS`. For a quicker cell, pass `n_trials=5`.

To **watch** TPE pick trials (toy landscape + live charts), open `notebooks/02_optuna_bayesian.ipynb` or Streamlit → Optuna lab.

In [6]:
opt = optimize_models(x_train_fe, x_test_fe, y_train, y_test)  # or n_trials=5
selection = select_best(
    model_metrics=opt.test_metrics,
    models=opt.best_models,
    feature_names=list(x_train_fe.columns),
    benchmark_metrics=bench.metrics,
    cv_scores=opt.cv_scores,
    best_params=opt.best_params,
    save=True,
)
display(selection.comparison)
print("winner:", selection.best_name)
selection.importances.head(15)

Optuna RF (15 trials)...


/Users/himanshujhamb/Desktop/Personal/Home_Assignment/Naresh/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(


Optuna XGBoost (15 trials)...


,model,pr_auc,roc_auc,f1,precision,recall,cv_pr_auc
0,xgboost,0.56,0.78,0.54,0.47,0.63,0.56
1,random_forest,0.55,0.78,0.54,0.49,0.60,0.56
2,logistic,0.49,0.73,0.50,0.41,0.64,NaN
3,majority,0.22,0.50,0.00,0.00,0.00,NaN


winner: xgboost


,feature,importance
0,MAX_PAY_STATUS,0.47
1,PAY_0,0.16
2,AVG_PAY_STATUS,0.05
3,UTIL_RATIO_1,0.02
4,PAY_2,0.02
5,LIMIT_BAL_bin,0.02
6,PAY_AMT3,0.02
7,PAY_AMT2,0.02
8,BILL_AMT6,0.01
9,PAY_RATIO_1,0.01


## One-liner alternative

```python
from credit_fraud.pipeline import run_training
result = run_training()  # or run_training(n_trials=5)
result.comparison
```